<a href="https://colab.research.google.com/github/menayatollahi-BAM/AIRF4201-AirlineManagement/blob/main/Module09/M09_Revenue_Management_Team_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIRF 4201 · Module 9 — Revenue Management Challenge
## You are the airline revenue-management team

**Goal:** Choose a management priority, then select how many early Basic bookings to accept, how many seats to leave available for late Flex requests, and whether to overbook. Test a strategy, examine the financial and service consequences, revise, and defend your recommendation.

**Time:** ~12 minutes in teams. **No calculators required.**

**How to run:** Click **Runtime → Run all** in Google Colab. Use the controls in the **Decision dashboard** cell. The Python source is collapsed by default; students do not need to open it.

**Decision rule:** You may not choose a policy that could deny boarding to **more than 3 people** in the modeled worst case. If your strategy violates this rule, it is not feasible.

**Version note:** This revised Colab case uses an updated four-outcome no-show forecast to ensure different airline priorities can favour different policies. Use the probabilities shown below.

This is a simplified **teaching simulation**, not operational booking advice. It assumes Basic customers are the first denied boarding, no-show counts apply to Basic passengers, all Flex passengers show up, denied passengers do not earn ticket or ancillary revenue, and denied boarding has an additional USD 900 cost.

### Case information (given to all teams)

| Input | Value |
|---|---:|
| Aircraft capacity | 180 seats |
| Basic fare (early demand 170) | USD 130 |
| Flex fare (late demand uncertain) | USD 290 |
| Flex high demand | 50 requests, probability 60% |
| Flex low demand | 20 requests, probability 40% |
| Basic ancillary revenue per flown passenger | USD 24 |
| Flex ancillary revenue per flown passenger | USD 8 |
| Basic no-shows | 3 (5%), 5 (15%), 8 (50%), or 10 (30%) |
| Additional cost per involuntary denial | USD 900 |
| Worst-case denied boarding policy limit | 3 |

**Updated simulation assumption:** The Basic no-show distribution has been revised to introduce a genuine trade-off between higher expected revenue and denied-boarding risk. These illustrative probabilities are part of this Colab case; use them instead of the earlier slide's three-outcome no-show forecast.

**Your decisions:** Select one **management priority** (Revenue Growth, Balanced Performance, or Customer Protection), plus Basic booking limit and overbooking authorization.

**Priority-adjusted score:** Expected revenue minus a decision penalty for the probability of at least one denied boarding. The penalty weight is **USD 0** (Revenue Growth), **USD 4,000** (Balanced Performance), or **USD 12,000** (Customer Protection), multiplied by that probability.

It is an illustrative managerial preference, **not a real cash expense**. Every team uses the same fares, USD 900 actual denial cost per passenger, probabilities and safety/service limit.

The simulator displays **both actual expected revenue and priority-adjusted score**, so you can see the trade-off.

The simulator infers late Flex capacity = **180 + overbooking allowance − Basic booking limit**.

**Before calculating:** Estimate mentally: Does raising overbooking from 0 to 6 help revenue? What service risk might it introduce?

In [ ]:
#@title Simulation model (code hidden) { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from math import isclose

CAPACITY = 180
BASIC_DEMAND = 170
FLEX_DEMAND = [(50, 0.60), (20, 0.40)]
BASIC_NO_SHOWS = [(3, 0.05), (5, 0.15), (8, 0.50), (10, 0.30)]
DENIAL_COST = 900
SERVICE_LIMIT = 3

def evaluate(basic_limit, overbook):
    # A basic booking limit is an accepted-booking CAP, not demand.
    basic = min(int(basic_limit), BASIC_DEMAND)
    total_sale_limit = CAPACITY + int(overbook)
    results = []
    for flex_requests, p_flex in FLEX_DEMAND:
        flex = min(flex_requests, max(0, total_sale_limit - basic))
        for no_shows, p_no in BASIC_NO_SHOWS:
            denied = max(0, basic + flex - no_shows - CAPACITY)
            # Case assumption: displaced/denied passengers come from Basic.
            basic_flown = max(0, basic - no_shows - denied)
            flex_flown = flex
            # A denied passenger's fare is refunded/not earned, plus penalty.
            revenue = basic_flown * (130 + 24) + flex_flown * (290 + 8) - DENIAL_COST * denied
            results.append({
                "flex_demand": flex_requests, "no_shows": no_shows,
                "prob": p_flex * p_no, "basic_sold": basic, "flex_sold": flex,
                "denied": denied, "flown": basic_flown + flex_flown,
                "revenue": revenue
            })
    expected = lambda key: sum(s["prob"] * s[key] for s in results)
    return {
        "expected_revenue": expected("revenue"),
        "expected_denied": expected("denied"),
        "chance_any_denied": sum(s["prob"] for s in results if s["denied"] > 0),
        "worst_denied": max(s["denied"] for s in results),
        "expected_flown": expected("flown"),
        "expected_basic_sold": expected("basic_sold"),
        "expected_flex_sold": expected("flex_sold"),
        "feasible": max(s["denied"] for s in results) <= SERVICE_LIMIT,
        "scenarios": results
    }

# Validate the demand distribution; the model uses eight joint scenarios.
assert abs(sum(p for _, p in FLEX_DEMAND) - 1.0) < 1e-10
assert abs(sum(p for _, p in BASIC_NO_SHOWS) - 1.0) < 1e-10
assert len(FLEX_DEMAND) * len(BASIC_NO_SHOWS) == 8

records = []
print("Simulator loaded. Use the team controls in the next cell.")

In [ ]:
#@title Decision dashboard (code hidden) { display-mode: "form" }
basic_limit = widgets.IntSlider(value=130, min=110, max=170, step=1, description="Basic limit:", continuous_update=False, style={"description_width":"110px"}, layout=widgets.Layout(width="580px"))
overbook = widgets.IntSlider(value=0, min=0, max=15, step=1, description="Overbook by:", continuous_update=False, style={"description_width":"110px"}, layout=widgets.Layout(width="580px"))
priority = widgets.Dropdown(
    options=[("Revenue Growth", "revenue"), ("Balanced Performance", "balanced"),
             ("Customer Protection", "protection")],
    value="balanced", description="Priority:",
    style={"description_width":"110px"}, layout=widgets.Layout(width="580px"))
# A management valuation of the chance of at least one denial, NOT an additional cash cost.
# All priorities retain the case's USD 900 per denied passenger and the same service rule.
PRIORITY_WEIGHTS = {"revenue": 0, "balanced": 4000, "protection": 12000}
PRIORITY_NAMES = {"revenue": "Revenue Growth", "balanced": "Balanced Performance",
                  "protection": "Customer Protection"}
def priority_score(result, selected):
    return result["expected_revenue"] - PRIORITY_WEIGHTS[selected] * result["chance_any_denied"]

# Silent instructor-side mathematical validation across the complete slider grid:
# The feasible optimum must differ for the three management priorities.
def _validate_decision_model():
    best_by_priority = {}
    for name, weight in PRIORITY_WEIGHTS.items():
        feasible = []
        for basic in range(basic_limit.min, basic_limit.max + 1, basic_limit.step):
            for over in range(overbook.min, overbook.max + 1, overbook.step):
                r = evaluate(basic, over)
                if r["feasible"]:
                    feasible.append((r["expected_revenue"] - weight * r["chance_any_denied"],
                                     basic, over, r))
        assert feasible, "No feasible strategies"
        # Prefer smaller overbooking allowances when decision scores tie.
        best_by_priority[name] = max(feasible, key=lambda t: (round(t[0], 6), -t[2], -t[1]))
    assert [best_by_priority[k][1] for k in ("revenue","balanced","protection")] == [136, 135, 133], (
        "The priorities must produce distinct optimal Basic booking limits."
    )
_validate_decision_model()

calculate = widgets.Button(description="Run my strategy", button_style="primary", icon="play", layout=widgets.Layout(width="175px"))
comparison = widgets.Button(description="Compare attempts", icon="line-chart")
reset = widgets.Button(description="Clear attempts", icon="trash")
output = widgets.Output()
records = []

def metric_card(label, value, subtitle="", color="#17365D"):
    return (f'<div style="flex:1;min-width:160px;background:#f3f6fb;border-radius:12px;'
            f'padding:15px;border-top:4px solid {color};">'
            f'<div style="color:#475569;font-size:13px;">{label}</div>'
            f'<div style="font-size:25px;font-weight:750;color:#17253a;margin:6px 0;">{value}</div>'
            f'<div style="font-size:12px;color:#64748b;">{subtitle}</div></div>')

def show_dashboard(row, result):
    compliance = result["feasible"]
    header = ('<div style="font-family:Arial,sans-serif;">'
              '<h3 style="margin-bottom:7px;">Your strategy results</h3>'
              f'<div style="margin-bottom:12px;color:#334155;">'
              f'Basic limit <b>{row["Basic limit"]}</b> · '
              f'Flex booking capacity <b>{row["Flex seats held"]}</b> · '
              f'Overbook allowance <b>{row["Overbook"]}</b> · '
              f'Management priority <b>{row["Priority"]}</b></div>')
    # The primary score comes first; the original cash revenue remains distinct.
    cards = "".join([
        metric_card("PRIORITY-ADJUSTED SCORE", f'USD {row["Priority score"]:,.0f}',
                    "Primary comparison metric for feasible strategies", "#4258a8"),
        metric_card("EXPECTED REVENUE", f'USD {result["expected_revenue"]:,.0f}',
                    "Expected cash revenue, after denial costs"),
        metric_card("PROBABILITY OF DENIAL", f'{result["chance_any_denied"]:.0%}',
                    "Chance of at least one denied boarding",
                    "#ba6a19" if result["chance_any_denied"] else "#247454"),
        metric_card("WORST-CASE DENIALS", str(result["worst_denied"]),
                    "Passengers denied in worst case",
                    "#b42318" if not compliance else "#247454"),
        metric_card("SERVICE RULE", "PASS" if compliance else "FAIL",
                    "Maximum 3 denials in any scenario",
                    "#247454" if compliance else "#b42318")
    ])
    display(HTML(header + '<div style="display:flex;flex-wrap:wrap;gap:10px;margin:12px 0 18px;">'
                 + cards + '</div></div>'))
    display(HTML('<p style="color:#475569;font-size:13px;">'
                 f'Priority-adjusted score = expected revenue − '
                 f'(USD {row["Priority weight"]:,} × probability of any denial). '
                 'The score is a decision measure, not accounting revenue.</p>'))
    if not compliance:
        display(HTML('<p style="color:#b42318;font-weight:700;">'
                     'Not eligible: this strategy violates the service constraint. Revise your policy.</p>'))
    else:
        display(HTML('<p style="color:#247454;font-weight:700;">'
                     'Eligible strategy. Compare its score with other feasible policies before recommending it.</p>'))

def show_comparison():
    if not records:
        print("No attempts yet. Run a strategy first.")
        return
    fig, ax = plt.subplots(figsize=(8.7, 4.1))
    for i, r in enumerate(records):
        color = "#267458" if r["Policy"] == "PASS" else "#c14d42"
        ax.scatter(r["P(denial)"]*100, r["Expected revenue"], s=150, c=color, alpha=.85,
                   edgecolors="white", linewidth=1.2)
        ax.annotate(f'Attempt {i+1} ({r["Priority"]})', (r["P(denial)"]*100,r["Expected revenue"]),
                    xytext=(5,7),textcoords="offset points", fontsize=8)
    ax.set_xlabel("Probability of any denied boarding (%)")
    ax.set_ylabel("Expected revenue (USD)")
    ax.set_title("Revenue vs passenger-service risk — team attempts", fontweight="bold", loc="left")
    ax.grid(alpha=.18)
    ax.margins(x=.15,y=.24)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    df=pd.DataFrame(records).copy()
    df.insert(0,"Attempt",range(1,len(df)+1))
    df["Expected revenue"]=df["Expected revenue"].map(lambda n:f"USD {n:,.0f}")
    df["P(denial)"]=df["P(denial)"].map(lambda n:f"{n:.0%}")
    df["Expected denials"]=df["Expected denials"].map(lambda n:f"{n:.2f}")
    df["Priority score"]=df["Priority score"].map(lambda n:f"USD {n:,.0f}")
    display(df[["Attempt","Basic limit","Overbook","Flex seats held",
                "Expected revenue","P(denial)","Worst denials","Priority","Priority score","Policy"]].style.hide(axis="index"))
    display(HTML('<div style="font-size:13px;color:#475569;">'
                 '<span style="color:#267458;">●</span> Compliant &nbsp; '
                 '<span style="color:#c14d42;">●</span> Violates service rule. '
                 'The simulator does not choose a winner: your team must justify its policy.</div>'))

def on_calculate(_):
    b, o = basic_limit.value, overbook.value
    p = priority.value
    result = evaluate(b, o)
    row = {"Basic limit": b,
           "Overbook": o, "Flex seats held": max(0, CAPACITY+o-b),
           "Priority": PRIORITY_NAMES[p], "Priority weight": PRIORITY_WEIGHTS[p],
           "Priority score": priority_score(result, p),
           "Expected revenue": result["expected_revenue"],
           "P(denial)": result["chance_any_denied"],
           "Worst denials": result["worst_denied"],
           "Expected denials": result["expected_denied"],
           "Policy": "PASS" if result["feasible"] else "FAIL"}
    records.append(row)
    with output:
        clear_output(wait=True)
        show_dashboard(row, result)
        if len(records)>1:
            print("Click 'Compare attempts' to see the revenue–risk trade-off for previous decisions.")

def on_compare(_):
    with output:
        clear_output(wait=True)
        show_comparison()

def on_reset(_):
    records.clear()
    with output:
        clear_output(wait=True)
        display(HTML("<p>Attempts cleared. Choose a new strategy and click Run my strategy.</p>"))

calculate.on_click(on_calculate)
comparison.on_click(on_compare)
reset.on_click(on_reset)
display(widgets.VBox([
    widgets.HTML("<h3>Choose your revenue-management policy</h3><p>Choose your management priority and change the sliders, then run the strategy. Do not seek the instructor's answer first.</p>"),
    priority, basic_limit, overbook, widgets.HBox([calculate, comparison, reset]), output
]))


### Team challenge · 12 minutes

1. **Round 1:** Choose a management priority and calculate one strategy. You may test a second policy after discussing why it should improve results.
2. **Round 2:** Compare your results with another team that has a different priority. Identify a strategy that produces **higher expected revenue** and one that creates **less denied-boarding risk**.
3. **Defend:** Recommend **one feasible policy** using expected revenue, probability of denial, worst-case denied passengers, and priority-adjusted score.

**Think and Respond:** If your strategy earns several hundred dollars more per flight but occasionally denies boarding, is the tradeoff acceptable? What if the next flight has no available seats for disrupted passengers?

